In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

candidates = [Path('titanic.csv'), Path('LAB1') / 'titanic.csv', Path('LAB1') / 'Bai2_Trang15-17' / 'titanic.csv', Path('../titanic.csv'), Path('/content/titanic.csv'), Path('/mnt/data/titanic.csv')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    try:
        from google.colab import files
    except ImportError:
        files = None
    if files is not None:
        print('Chưa tìm thấy titanic.csv. Hãy chọn tệp CSV để tải lên Colab.')
        uploaded = files.upload()
        uploaded_path = next((Path(name) for name in uploaded if Path(name).name.lower() == 'titanic.csv'), None)
        if uploaded_path is not None and uploaded_path.is_file():
            data_path = uploaded_path
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy titanic.csv. Thư mục chạy hiện tại: {Path.cwd()}. Hãy tải CSV lên Colab hoặc đặt CSV trong thư mục này rồi chạy lại ô.')
df = pd.read_csv(data_path)
df['age'] = df['age'].fillna(df['age'].median())
df['fare'] = df['fare'].fillna(df['fare'].median())
if df['embarked'].isna().any():
    df['embarked'] = df['embarked'].fillna(df['embarked'].mode().iloc[0])

df['title'] = df['name'].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
fare_precise = df['fare'].copy()
df['fare'] = df['fare'].astype(int)
df['sex'] = df['sex'].map({'male': 0, 'female': 1}).astype('int64')
df['embarked'] = df['embarked'].map({'S': 0, 'C': 1, 'Q': 2}).astype('int64')

df['age_group'] = pd.qcut(df['age'].rank(method='first'), q=8, labels=False) + 1
age_bins = [-float('inf'), 11, 18, 22, 27, 33, 40, 66, float('inf')]
df['age_group_suggested'] = pd.cut(df['age'], bins=age_bins, labels=False, include_lowest=True) + 1
df['fare_group'] = pd.qcut(df['fare'].rank(method='first'), q=6, labels=False) + 1
fare_bins = [-float('inf'), 7.91, 14.454, 31, 99, 250, float('inf')]
df['fare_group_suggested'] = pd.cut(fare_precise, bins=fare_bins, labels=False, include_lowest=True) + 1

print('Võ Thiện Linh - 6451071041')
print('Bảng mã: sex {male: 0, female: 1}; embarked {S: 0, C: 1, Q: 2}')
print('Phân bố nhóm tuổi cân bằng:')
print(df['age_group'].value_counts().sort_index().to_string())
print('\nPhân bố nhóm giá vé cân bằng:')
print(df['fare_group'].value_counts().sort_index().to_string())
print('\nPhân bố nhóm theo ngưỡng tuổi gợi ý:')
print(df['age_group_suggested'].value_counts().sort_index().to_string())
print('\nPhân bố nhóm theo ngưỡng giá vé gợi ý:')
print(df['fare_group_suggested'].value_counts().sort_index().to_string())
print('\nKiểu dữ liệu fare:', df['fare'].dtype)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 48)
print('\nBảng 1. Tên và các thuộc tính phân loại (tên cột gốc; 10 dòng đầu)')
display(df[['name', 'title', 'sex', 'embarked']].head(10))
print('\nBảng 2. Tuổi và các cột nhóm tuổi (tên cột gốc; 10 dòng đầu)')
display(df[['age', 'age_group', 'age_group_suggested']].head(10))
print('\nBảng 3. Giá vé và các cột nhóm giá vé (tên cột gốc; 10 dòng đầu)')
display(df[['fare', 'fare_group', 'fare_group_suggested']].head(10))